### Build a Question Answering application over a Graph Database

In [38]:
NEO4J_URI="XXXXXXXXXXXXXXX"
NEO4J_USERNAME="xxxxxxxxxxxxxxxx"
NEO4J_PASSWORD="xxxxxxxxxxxxxxxx"
NEO4J_DATABASE="xxxxxxxxxxxxxxxx"

In [39]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE

In [40]:
pip install langchain_community

Note: you may need to restart the kernel to use updated packages.


In [41]:
pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [42]:
print("URI:", NEO4J_URI)
print("Username:", NEO4J_USERNAME)

URI: XXXXXXXXXXXXXXX
Username: xxxxxxxxxxxxxxxx


In [43]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username="xxxxx",
    password=NEO4J_PASSWORD,
    database="xxxx"
)

ConfigurationError: URI scheme '' is not supported. Supported URI schemes are ['bolt', 'bolt+ssc', 'bolt+s', 'neo4j', 'neo4j+ssc', 'neo4j+s']. Examples: bolt://host[:port] or neo4j://host[:port][?routing_context]

In [ ]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
graph

In [ ]:
## Dataset Movie
movie_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row
MERGE (m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbrating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""




In [ ]:
movie_query

"\nLOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row\nMERGE (m:Movie{id:row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbrating)\nFOREACH (director in split(row.director, '|') |\n    MERGE (p:Person {name:trim(director)})\n    MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor in split(row.actors, '|') |\n    MERGE (p:Person {name:trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre in split(row.genres, '|') |\n    MERGE (g:Genre {name:trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g))\n"

In [ ]:
graph.query(movie_query)

[]

In [ ]:
print(movie_query)


LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row
MERGE (m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbrating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))



In [ ]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [ ]:
pip install langchain_groq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_groq import ChatGroq
llm= ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile")

In [ ]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016D5CDF8190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016D5CDF8B90>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import langchain
print(langchain.__version__)

1.3.11


In [ ]:
pip install -U langchain langchain-community langchain-neo4j

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

In [ ]:
response = chain.invoke(
    {"query": "Who directed Toy Story?"}
)

print(response)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Toy Story'}) RETURN p.name
Full Context:
[{'p.name': 'John Lasseter'}]

> Finished chain.
{'query': 'Who directed Toy Story?', 'result': 'John Lasseter directed Toy Story.'}


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key
)

print(llm.invoke("Hello"))

content='Hello. How can I help you today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.036055386, 'completion_tokens_details': None, 'prompt_time': 0.003690169, 'prompt_tokens_details': None, 'queue_time': 0.161808541, 'total_time': 0.039745555}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f1cab-bfe7-7ee1-bcc7-f7f253a64b55-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46}


In [ ]:
response = chain.invoke(
    {"query": "Find the actor with the highest number of movies in the database."}
)

print(response)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p.name, COUNT(m) AS num_movies ORDER BY num_movies DESC LIMIT 1
Full Context:
[{'p.name': 'Gene Hackman', 'num_movies': 4}]

> Finished chain.
{'query': 'Find the actor with the highest number of movies in the database.', 'result': 'Gene Hackman is the actor with the highest number of movies, having been in 4 movies.'}
